In [8]:
import pandas as pd
import datetime

import time
from mlflow.tracking.client import MlflowClient
from mlflow.entities.model_registry.model_version_status import ModelVersionStatus
from mlflow.tracking.client import MlflowClient
from mlflow.tracking import MlflowClient
import mlflow.pyfunc

import json

In [14]:
client = MlflowClient('file:./mlruns')

# Constants used:
current_date = datetime.datetime.now().strftime("%Y_%B_%d")
artifact_path = "model"
model_name = "lead_model"
experiment_name = current_date

In [10]:
def wait_until_ready(model_name, model_version):
    client = MlflowClient()
    for _ in range(10):
        model_version_details = client.get_model_version(
          name=model_name,
          version=model_version,
        )
        status = ModelVersionStatus.from_string(model_version_details.status)
        print(f"Model status: {ModelVersionStatus.to_string(status)}")
        if status == ModelVersionStatus.READY:
            break
        time.sleep(1)

In [ ]:

experiment_ids = [client.get_experiment_by_name(experiment_name).experiment_id]

AttributeError: 'NoneType' object has no attribute 'experiment_id'

In [ ]:
experiment_best = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["metrics.f1_score DESC"],
    max_results=1
).iloc[0]

In [ ]:
with open("./artifacts/model_results.json", "r") as f:
    model_results = json.load(f)

results_df = pd.DataFrame({model: val["weighted avg"] for model, val in model_results.items()}).T

In [ ]:
best_model = results_df.sort_values("f1-score", ascending=False).iloc[0].name

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
prod_model = [model for model in client.search_model_versions(f"name='{model_name}'") if dict(model)['current_stage']=='Production']
prod_model_exists = len(prod_model)>0

if prod_model_exists:
    prod_model_version = dict(prod_model[0])['version']
    prod_model_run_id = dict(prod_model[0])['run_id']
    
    print('Production model name: ', model_name)
    print('Production model version:', prod_model_version)
    print('Production model run id:', prod_model_run_id)
    
else:
    print('No model in production')


In [ ]:
train_model_score = experiment_best["metrics.f1_score"]
model_details = {}
model_status = {}
run_id = None

if prod_model_exists:
    data, details = mlflow.get_run(prod_model_run_id)
    prod_model_score = data[1]["metrics.f1_score"]

    model_status["current"] = train_model_score
    model_status["prod"] = prod_model_score

    if train_model_score>prod_model_score:
        print("Registering new model")
        run_id = experiment_best["run_id"]
else:
    print("No model in production")
    run_id = experiment_best["run_id"]

print(f"Registered model: {run_id}")

In [ ]:
if run_id is not None:
    print(f'Best model found: {run_id}')

    model_uri = "runs:/{run_id}/{artifact_path}".format(
        run_id=run_id,
        artifact_path=artifact_path
    )
    model_details = mlflow.register_model(model_uri=model_uri, name=model_name)
    wait_until_ready(model_details.name, model_details.version)
    model_details = dict(model_details)
    print(model_details)

In [ ]:
MlflowClient()

In [1]:
import mlflow
print(f"Current Tracking URI: {mlflow.get_tracking_uri()}")

Current Tracking URI: http://72.145.4.225:5000


In [23]:
# Run this command in a separate terminal window
!mlflow ui --port 5000

[2025-11-10 10:08:45 +0100] [16767] [INFO] Starting gunicorn 23.0.0
[2025-11-10 10:08:45 +0100] [16767] [INFO] Listening at: http://127.0.0.1:5000 (16767)
[2025-11-10 10:08:45 +0100] [16767] [INFO] Using worker: sync
[2025-11-10 10:08:45 +0100] [16768] [INFO] Booting worker with pid: 16768
[2025-11-10 10:08:45 +0100] [16769] [INFO] Booting worker with pid: 16769
[2025-11-10 10:08:45 +0100] [16770] [INFO] Booting worker with pid: 16770
[2025-11-10 10:08:45 +0100] [16771] [INFO] Booting worker with pid: 16771
2025/11/10 10:09:09 ERROR mlflow.server: Exception on /model-versions/get-artifact [GET]
Traceback (most recent call last):
  File "/Users/sunechristiansen/anaconda3/envs/my_project_env/lib/python3.11/site-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sunechristiansen/anaconda3/envs/my_project_env/lib/python3.11/site-packages/flask/app.py", line 919, in full_dispatch_request
    r

In [ ]:
import mlflow


# This explicitly tells MLflow to use the local 'mlruns' folder
mlflow.set_tracking_uri("file:./mlruns")

# Now try running your code again
with mlflow.start_run():
    mlflow.log_param("alpha", 0.01)
    mlflow.log_metric("loss", 0.5)

In [5]:
client = MlflowClient()

NameError: name 'MlflowClient' is not defined

In [ ]:
client.search_experiments()


[<Experiment: artifact_location='file:///Users/sunechristiansen/sune/MLops_project/itu-MLOps-project/project/project_repo/notebooks/mlruns/0', creation_time=1762763226516, experiment_id='0', last_update_time=1762763226516, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
client.get_model_version()

NameError: name 'client' is not defined

In [ ]:
import mlflow
import datetime
import json
import os
import pandas as pd
import time
from mlflow.tracking import MlflowClient
from mlflow.entities.model_registry.model_version_status import ModelVersionStatus

# --- 1. Define Constants and Client (as per your script) ---

# Constants used:
current_date = datetime.datetime.now().strftime("%Y_%B_%d")
artifact_path = "model"
model_name = "lead_model"
experiment_name = current_date # e.g., "2025_November_10"
DUMMY_F1_SCORE = 0.95 # This will be the "best" score for the search
DUMMY_BEST_MODEL_NAME = "XGBoost_Model" # This will be the best_model name

mlflow.set_tracking_uri("file:./mlruns")
client = MlflowClient()


# --- 2. Create Required Artifacts (Files) ---

# 2.1. Ensure the artifacts directory exists
ARTIFACTS_DIR = "./artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f"Created directory: {ARTIFACTS_DIR}")




# --- 3. Create MLflow Experiment and Run ---

# 3.1. Create the experiment (or get existing ID)
try:
    experiment_id = client.create_experiment(experiment_name)
    print(f"Created new experiment: {experiment_name} (ID: {experiment_id})")
except Exception:
    # Handle case where experiment already exists
    experiment_id = client.get_experiment_by_name(experiment_name).experiment_id
    print(f"Using existing experiment: {experiment_name} (ID: {experiment_id})")


# 3.2. Start and log a dummy run
with mlflow.start_run(experiment_id=experiment_id, run_name=DUMMY_BEST_MODEL_NAME) as run:
    dummy_run_id = run.info.run_id
    
    # Log the required metric (f1_score) to satisfy the search_runs filter
    mlflow.log_metric("f1_score", DUMMY_F1_SCORE)
    
    # Log the required artifact (the model itself) to satisfy model_uri
    mlflow.log_artifact(DUMMY_MODEL_FILE, artifact_path=artifact_path)
    
    # Log the best_model name as a tag, which might be useful if you search for it later
    mlflow.set_tag("best_model_name", DUMMY_BEST_MODEL_NAME)
    
    print("-" * 50)
    print(f"Successfully logged dummy run: {dummy_run_id}")
    print(f"Logged Metric: f1_score = {DUMMY_F1_SCORE}")
    print(f"Logged Artifact at: runs:/{dummy_run_id}/{artifact_path}")
    print("-" * 50)

# --- 4. Run your original deployment script here ---
print("\n" * 2)
print("=" * 10 + " Starting Execution of Your Original Deployment Code " + "=" * 10)
print("\n" * 2)

# Now, you can paste and run your original code (starting from the wait_until_ready function)
# or just include it here.

# --- Original Deployment Code (Modified slightly for context) ---

# Note: The wait_until_ready function and imports are already in the environment

def wait_until_ready(model_name, model_version):
    client = MlflowClient()
    for _ in range(10):
        model_version_details = client.get_model_version(
          name=model_name,
          version=model_version,
        )
        status = ModelVersionStatus.from_string(model_version_details.status)
        print(f"Model status: {ModelVersionStatus.to_string(status)}")
        if status == ModelVersionStatus.READY:
            break
        time.sleep(1)

# Re-initialize client (not strictly needed but good practice)
client = MlflowClient()

experiment_ids = [client.get_experiment_by_name(experiment_name).experiment_id]

# This search now finds the dummy run
experiment_best = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["metrics.f1_score DESC"],
    max_results=1
).iloc[0]

# This file read now succeeds
with open(os.path.join(ARTIFACTS_DIR, "model_results.json"), "r") as f:
    model_results = json.load(f)

results_df = pd.DataFrame({model: val["weighted avg"] for model, val in model_results.items()}).T

# This now finds the DUMMY_BEST_MODEL_NAME
best_model = results_df.sort_values("f1-score", ascending=False).iloc[0].name
print(f"Model identified as 'best' from results file: {best_model}")

# Check Model Registry
prod_model = [model for model in client.search_model_versions(f"name='{model_name}'") if dict(model)['current_stage']=='Production']
prod_model_exists = len(prod_model)>0

if prod_model_exists:
    prod_model_version = dict(prod_model[0])['version']
    prod_model_run_id = dict(prod_model[0])['run_id']
    print('Production model name: ', model_name)
    print('Production model version:', prod_model_version)
    print('Production model run id:', prod_model_run_id)
else:
    print('No model in production (as expected for the first run)')


train_model_score = experiment_best["metrics.f1_score"]
model_details = {}
model_status = {}
run_id = None

if prod_model_exists:
    # This block won't run on the first execution
    pass
else:
    print("No model in production. Proceeding to register the new best run.")
    run_id = experiment_best["run_id"]

print(f"Run ID for model registration: {run_id}")

if run_id is not None:
    print(f'Best model found: {run_id}')

    model_uri = "runs:/{run_id}/{artifact_path}".format(
        run_id=run_id,
        artifact_path=artifact_path
    )
    # This is the line that creates the first entry in the Model Registry
    model_details = mlflow.register_model(model_uri=model_uri, name=model_name)
    
    print(f"Registered model version: {model_details.version}")
    wait_until_ready(model_details.name, model_details.version)
    
    # You can now transition this model to Staging or Production
    client.transition_model_version_stage(
        name=model_details.name,
        version=model_details.version,
        stage="Production" # Set the first model directly to Production
    )
    print(f"Transitioned Model Version {model_details.version} to Production.")

print("=" * 10 + " Deployment Code Execution Complete " + "=" * 10)

Created directory: ./artifacts
Created required file: ./artifacts/model_results.json
Using existing experiment: 2025_November_10 (ID: 567643722203702435)
--------------------------------------------------
Successfully logged dummy run: 86fa565cd43d4a8ea1329dc944dad08f
Logged Metric: f1_score = 0.95
Logged Artifact at: runs:/86fa565cd43d4a8ea1329dc944dad08f/model
--------------------------------------------------



========== Starting Execution of Your Original Deployment Code ==========



Model identified as 'best' from results file: XGBoost_Model
No model in production (as expected for the first run)
No model in production. Proceeding to register the new best run.
Run ID for model registration: 86fa565cd43d4a8ea1329dc944dad08f
Best model found: 86fa565cd43d4a8ea1329dc944dad08f
Registered model version: 1
Model status: READY
Transitioned Model Version 1 to Production.
========== Deployment Code Execution Complete ==========


Successfully registered model 'lead_model'.
Created version '1' of model 'lead_model'.
/var/folders/vw/nff8_41x38jcbpv0nvrcxv700000gn/T/ipykernel_9305/3745573942.py:179: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [22]:
model_version = 1

def wait_for_deployment(model_name, model_version, stage='Staging'):
    status = False
    while not status:
        model_version_details = dict(
            client.get_model_version(name=model_name,version=model_version)
            )
        if model_version_details['current_stage'] == stage:
            print(f'Transition completed to {stage}')
            status = True
            break
        else:
            time.sleep(2)
    return status

model_version_details = dict(client.get_model_version(name=model_name,version=model_version))
model_status = True
if model_version_details['current_stage'] != 'Staging':
    client.transition_model_version_stage(
        name=model_name,
        version=model_version,stage="Staging", 
        archive_existing_versions=True
    )
    model_status = wait_for_deployment(model_name, model_version, 'Staging')
else:
    print('Model already in staging')

Transition completed to Staging


/var/folders/vw/nff8_41x38jcbpv0nvrcxv700000gn/T/ipykernel_9305/1598305847.py:20: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
